In [4]:
## TODO: All project members will need to pip install rasterio 

import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import rasterio
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, accuracy_score
from torchvision import models
from torch import optim
from torchmetrics.classification import MulticlassConfusionMatrix
import torch.nn.functional as F


DIR_PATH = "../EuroSATallBands/"

#dataloader class
class EuroSATDataset(Dataset):
    def __init__(self, csv_file):
        df = pd.read_csv(DIR_PATH + csv_file)

        self.files = df.iloc[:, 0].tolist()
        #make sure the class IDs are parsed as an int
        self.labels = df.iloc[:, 1].astype(int).tolist()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        filename = DIR_PATH + self.files[index]
        #must use rasterio to import the satellite TIF
        with rasterio.open(filename) as img:
            image = img.read().astype(np.float32)

        X = torch.from_numpy(image)
        #this is needed for CrossEntropyLoss later
        Y = torch.tensor(self.labels[index], dtype = torch.int)

        return X, Y

#default parameters: batch size 32, no max, do not shuffle, no workers, random seed of 2 for consistency
def make_dataloader(csv_file, batchsize=32, max_samples = None, shuffle = False, numWorkers = 0, seed = 2):
    r"""
    A function to create a Torch dataloader on a dataset or a subset of a dataset.

    Args:
        csv_file (string): The name of the csv file to be read. Expected format is filepath, classID, className
        batchsize (integer): The size of the batches to load
        max_samples (integer): Default(None) - How many samples to include in the dataloader
        shuffle (boolean): Default(False) - Whether or not the dataloader entires should be shuffled
        numWorkers (integer): Default(0) - How many concurrent workers should be run
        seed(integer): Default(2) - The RNG seed

    """
    dataset = EuroSATDataset(csv_file)
    #This is for splitting the data into smaller subsets so we don't have to train on the whole dataset
    if max_samples is not None:
        if max_samples < 1:
            raise ValueError("Maximum number of records must be positive")

        #for consistency in comparing models
        rng = torch.Generator().manual_seed(seed)
        indices = torch.randperm(dataset.__len__(), generator = rng)[:max_samples].tolist()

        dataset = Subset(dataset, indices)

    return DataLoader(dataset,
                      batch_size = batchsize, 
                      shuffle = shuffle, 
                      num_workers = numWorkers, 
                      pin_memory = torch.cuda.is_available(), 
                      persistent_workers = numWorkers > 0)


In [5]:
# small dataset for model testing
train_loader = make_dataloader(
    "train.csv",
    batchsize=32,
    max_samples=500,
    shuffle=True
)

valid_loader = make_dataloader(
    "validation.csv",
    batchsize=32,
    max_samples=100
)

test_loader = make_dataloader(
    "test.csv",
    batchsize=32,
)
# full dataset
# train_loader = make_dataloader(
#     "train.csv",
#     batchsize=32,
#     shuffle=True
# )

# valid_loader = make_dataloader(
#     "validation.csv",
#     batchsize=32,
# )

# test_loader = make_dataloader(
#     "test.csv",
#     batchsize=32,
# )

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [7]:
class model(torch.nn.Module):
    def __init__(self):
        super(model, self).__init__() # Superclass constructor

        self.conv1 = torch.nn.Conv2d(in_channels=13, out_channels=6, kernel_size=5, stride=1, padding='same')
        self.pool1 = torch.nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = torch.nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding='same')
        self.pool2 = torch.nn.AvgPool2d(kernel_size=2, stride=2)
        self.fc1 = torch.nn.Linear(in_features=16*16*16, out_features=120)
        self.fc2 = torch.nn.Linear(in_features=120, out_features=84)
        self.fc3 = torch.nn.Linear(in_features=84, out_features=10)

    def forward(self, X):
        a1 = F.relu(self.conv1(x)) 
        a1p = self.pool1(a1)            
        a2 = F.relu(self.conv2(a1p))    
        a2p = self.pool2(a2)
        flat = a2p.view(a2p.shape[0], 16*16*16)
        a3 = F.relu(self.fc1(flat))      
        a4 = F.relu(self.fc2(a3))      
        z5 = self.fc3(a4)
        return z5


In [8]:
SatelliteImageClassifier = model()
#make sure our whole model runs on same device
atelliteImageClassifier = SatelliteImageClassifier.to(device)
costFunction = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(SatelliteImageClassifier.parameters(), lr = 0.001)

In [9]:
def train_model(model, trainloader, costFunction, optimizer, numEpochs):
    for epoch in tqdm(range(numEpochs),desc="Training",unit="Epoch"):
        model.train()
        totalLoss = 0
        for satimgs, labels in trainloader:
            satimgs = satimgs.to(device)
            labels = labels.to(device).long()

            #forward
            outputs = model(satimgs)
            loss = costFunction(outputs, labels)

            #backpropigation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            #keep track of loss
            totalLoss += loss.item()
        
        #display loss every epoch
        avg_loss = totalLoss / len(train_loader)
        if epoch % 1 == 0:
            print("Loss " + str(avg_loss))
        

In [10]:
train_model(SatelliteImageClassifier, train_loader, costFunction, optimizer, 2)

Training:   0%|          | 0/2 [00:01<?, ?Epoch/s]


NameError: name 'x' is not defined